In [4]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage


class ChatOpenAIWithReasoning(ChatOpenAI):
    """保留第三方接口返回的 reasoning_content。

    ChatOpenAI 只按 OpenAI 官方规范解析响应，会丢弃 reasoning_content
    这类非标准字段。这里在解析完成后，把原始响应里的推理内容补回到
    additional_kwargs 中。
    """

    def _create_chat_result(self, response, generation_info=None):
        result = super()._create_chat_result(response, generation_info)

        response_dict = (
            response if isinstance(response, dict) else response.model_dump(warnings=False)
        )
        choices = response_dict.get("choices") or []

        for generation, choice in zip(result.generations, choices):
            reasoning_content = choice.get("message", {}).get("reasoning_content")
            if reasoning_content:
                generation.message.additional_kwargs["reasoning_content"] = (
                    reasoning_content
                )

        return result

In [6]:
llm = ChatOpenAIWithReasoning(
    temperature=0.2,
    extra_body={"reasoning_effort": "low"},
    model=os.environ["LLM_MODEL"],
    api_key=os.environ["LLM_API_KEY"],
    base_url=os.environ["LLM_BASE_URL"],
)

# 思考过程

In [8]:
from IPython.display import display, Markdown

messages = [
    HumanMessage(content="请帮我解方程：x^2 - 5x + 6 = 0")
]
result = llm.invoke(messages)

reasoning = result.additional_kwargs.get("reasoning_content", "")
final_answer = result.content

In [9]:
# 渲染思考过程
print("=== 思考过程 ===")
if reasoning:
    display(Markdown(reasoning))  # 加个引用符号区分
else:
    print("无思考过程")

print("\n=== 最终回答 ===")
display(Markdown(final_answer))

=== 思考过程 ===


用户要求解一个二次方程：x² - 5x + 6 = 0

这是一个标准的二次方程，可以用多种方法求解：
1. 因式分解法
2. 求根公式法
3. 配方法

最简便的是因式分解法：
x² - 5x + 6 = 0
需要找两个数，乘积为6，和为-5
这两个数是 -2 和 -3
所以 (x - 2)(x - 3) = 0
x = 2 或 x = 3

我用因式分解法来解答，简洁明了。


=== 最终回答 ===


# 解方程：x² - 5x + 6 = 0

## 方法：因式分解

寻找两个数，使得：
- 乘积 = 6
- 和 = -5

这两个数是 **-2** 和 **-3**（因为 (-2)×(-3) = 6，(-2)+(-3) = -5）

因此方程可以分解为：

$$(x - 2)(x - 3) = 0$$

## 求解

令每个因子为零：

$$x - 2 = 0 \quad \Rightarrow \quad x = 2$$
$$x - 3 = 0 \quad \Rightarrow \quad x = 3$$

## 结果

$$\boxed{x_1 = 2, \quad x_2 = 3}$$

# 结构化输出

In [10]:
from pydantic import BaseModel, Field


class TestSchema(BaseModel):
    reasoning: str = Field(description="简短的思考过程")
    verification_token: str = Field(description="必须填写为 'CONFIRMED_BY_SCHEMA'，不要改变它")
    answer: str = Field(description="最终的答案")
    model_name: str = Field(default="qwen-plus", description="模型名称")

In [11]:
messages = [
    HumanMessage(content="你是一个AI助手，请随便生成一个 JSON 数据"),
]
result = llm.invoke(messages, response_format={"type": "json_object"})
result.pretty_print()

================================== Ai Message ==================================

{
  "name": "张三",
  "age": 28,
  "city": "北京",
  "hobbies": [
    "读书",
    "编程",
    "旅行"
  ],
  "is_active": true,
  "score": 95.5
}


In [12]:
messages = [
    HumanMessage(content=f"你是一个AI助手，请随便生成一个 JSON 数据。"),
]
result = llm.invoke(
    messages,
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "schema_name",
            "schema": TestSchema.model_json_schema()
        },
    }
)
result.pretty_print()

================================== Ai Message ==================================

[{"reasoning": "用户要求生成一个符合指定JSON Schema的随机数据。根据Schema，需要包含reasoning（思考过程）、verification_token（固定值）、answer（最终答案）和model_name（默认qwen-plus）。我将生成一个符合要求的JSON对象。", "verification_token": "CONFIRMED_BY_SCHEMA", "answer": "这是一个随机生成的JSON数据示例，包含了一个简单的用户信息对象。", "model_name": "qwen-plus"}]
